To load data (read it and make it accessible) is often the 1st step in data analysis. To 'parse' it also means to load text data and interpret it as some data type, eg a table. Data input and output means to read .txt files or more efficient on-disk formats, load from databases, and interact with network sources, eg APIs. While many python tools help with this, our focus is on pd.<br>
pd functions to read tabular data as a <i>pd.DataFrame</i>
<table border="1">
<tr><th>Function</th><th>Description</th></tr>
<tr><td>read_csv</td>
<td>Load delimited data from a file, URL, or file-like object; use comma as default delimiter</td></tr>
<tr><td>read_fwf</td>
<td>Read data in fixed-width column format (i.e., no delimiters)</td></tr>
<tr><td>read_clipboard</td>
<td>Variation of read_csv that reads data from the clipboard; useful for converting tables from web pages</td></tr>
<tr><td>read_excel</td>
<td>Read tabular data from an Excel XLS or XLSX file</td></tr>
<tr><td>read_hdf</td>
<td>Read HDF5 files written by pandas</td></tr>
<tr><td>read_html</td>
<td>Read all tables found in the given HTML document</td></tr>
<tr><td>read_json</td>
<td>Read data from a JSON (JavaScript Object Notation) string representation,<br>file, URL, or file-like object</td></tr>
<tr><td>read_feather</td>
<td>Read the Feather binary file format</td></tr>
<tr><td>read_orc</td>
<td>Read the Apache ORC binary file format</td></tr>
<tr><td>read_parquet</td>
<td>Read the Apache Parquet binary file format</td></tr>
<tr><td>read_pickle</td>
<td>Read an object stored by pandas using the Python pickle format</td></tr>
<tr><td>read_sas</td>
<td>Read a SAS dataset stored in one of the SAS system's custom storage formats</td></tr>
<tr><td>read_spss</td>
<td>Read a data file created by SPSS</td></tr>
<tr><td>read_sql</td>
<td>Read the results of a SQL query (using SQLAlchemy)</td></tr>
<tr><td>read_sql_table</td>
<td>Read a whole SQL table (using SQLAlchemy); equivalent to using a query that selects<br>everything in that table using read_sql</td></tr>
<tr><td>read_stata</td>
<td>Read a dataset from Stata file format</td></tr>
<tr><td>read_xml</td>
<td>Read a table of data from an XML file</td></tr>
</table>

Most of the above functions' optional arguments fall into the categories
<ul><li>Indexing: treat 1+ columns as the (index of) the returned <i>pd.DataFrame</i>, and whether to get column names from the file, as the user provides, or not at all
<li>Type inference/ data conversion: also allows a custom list of missing value markers
<li>Date/ time parsing: can combine date/ time info spread over many columns into just 1 column
<li>Iterating: over chunks of big files
<li>Unclean data issues: eg skip rows or a footer, comments, numeric data with commas</ul>
Because of how messy real world data often is, many of the above functions have accumulated long lists of optional arguments over time (<i>pd.read_csv</i> alone has 50+). It is normal to feel overwhelmed

In [ ]:
import csv
import json
from lxml import objectify #cannot import lxml; lmxl.objectify
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import requests
import sqlalchemy as sqla
import sqlite3
import sys

In [ ]:
github_start='https://github.com/wesm/pydata-book/raw/refs/heads/3rd-edition/'
github_start2='https://raw.githubusercontent.com/wesm/pydata-book/refs/heads/3rd-edition/'

In [ ]:
#!cat examples/ex1.csv
!curl -sL {github_start+'examples/ex1.csv'}

In [ ]:
#top row becomes column names, default index 0,1,etc
pd.read_csv(github_start+"examples/ex1.csv")

In [ ]:
!curl -sL {github_start+'examples/ex2.csv'} #no header row

In [ ]:
#pd assigns column names 0,1,etc
pd.read_csv(github_start+"examples/ex2.csv", header=None)
cn=["a", "b", "c", "d", "message"]
pd.read_csv(github_start+"examples/ex2.csv", names=cn) #give column names
#make a column the index (same: index_col="message")
pd.read_csv(github_start+"examples/ex2.csv", names=cn, index_col=4)

In [ ]:
!curl -sL {github_start+'examples/csv_mindex.csv'}
#make a hierarchical index
pd.read_csv(github_start+"examples/csv_mindex.csv",index_col=["key1", "key2"])

Fields separated by variable amount of whitespace, motivates the regexp '\s+' as <i>sep</i>. Also, the top row has 1 less column name than number of columns, so <i>pd.read_csv</i> infers the leading column should be the index

In [ ]:
!curl -sL {github_start+'examples/ex3.csv'}
pd.read_csv(github_start+"examples/ex3.txt", sep="\s+")

In [ ]:
!curl -sL {github_start+'examples/ex4.csv'}
pd.read_csv(github_start+"examples/ex4.csv", skiprows=[0, 2, 3])

Handling missing values is important and often nuanced. They are often marked by empty strings, or sentinel (placeholder) values, eg NA or NULL. pd handles these by default. Give an iterable to <i>na_values</i> for what additional strings (on top of the defaults) to treat as missing. Disable the defaults with <i>keep_default_na=False</i>. <i>na_values</i> can also be a <i>dict</i>, allowing different NA sentinels per column

In [ ]:
!curl -sL {github_start+'examples/ex5.csv'}
qjx=pd.read_csv(github_start+"examples/ex5.csv")
pd.isna(qjx)
qjx = pd.read_csv(github_start+"examples/ex5.csv", na_values=["NULL"])
qjx = pd.read_csv(github_start+"examples/ex5.csv", keep_default_na=False)
qjx.isna() #all False
qjx = pd.read_csv(github_start+"examples/ex5.csv", keep_default_na=False,na_values=["NA"])
qjx.isna()
sentinels = {"message": ["foo", "NA"], "something": ["two"]}
pd.read_csv(github_start+"examples/ex5.csv", na_values=sentinels,keep_default_na=False)

summary of <i>pd.read_csv</i> options
<table border="1">
<tr><th>Argument</th><th>Description</th></tr>
<tr><td>path</td>
<td>String indicating filesystem location, URL, or file-like object.</td></tr>
<tr><td>sep or delimiter</td>
<td>Character sequence or regular expression to use to split fields in each row.</td></tr>
<tr><td>header</td>
<td>Row number to use as column names; defaults to 0 (first row),<br>but should be None if there is no header row.</td></tr>
<tr><td>index_col</td>
<td>Column numbers or names to use as the row index in the result;<br>can be a single name/number or a list for a hierarchical index.</td></tr>
<tr><td>names</td>
<td>List of column names for result.</td></tr>
<tr><td>skiprows</td>
<td>Number of rows at beginning of file to ignore or list of row numbers<br>(starting from 0) to skip.</td></tr>
<tr><td>na_values</td>
<td>Sequence of values to replace with NA. Added to default list unless<br>keep_default_na=False is passed.</td></tr>
<tr><td>keep_default_na</td>
<td>Whether to use the default NA value list or not (True by default).</td></tr>
<tr><td>comment</td>
<td>Character(s) to split comments off the end of lines.</td></tr>
<tr><td>parse_dates</td>
<td>Attempt to parse data to datetime; False by default. If True, parses all columns.<br>Otherwise specify list of columns; can combine multiple columns into one date.</td></tr>
<tr><td>keep_date_col</td>
<td>If joining columns to parse date, keep the joined columns;<br>False by default.</td></tr>
<tr><td>converters</td>
<td>Dictionary mapping column name/number to functions<br>(e.g., {"foo": f} applies f to all values in column "foo").</td></tr>
<tr><td>dayfirst</td>
<td>When parsing ambiguous dates, treat as international format<br>(e.g., 7/6/2012 → June 7, 2012); False by default.</td></tr>
<tr><td>date_parser</td>
<td>Function to use to parse dates.</td></tr>
<tr><td>nrows</td>
<td>Number of rows to read from beginning of file (excluding header).</td></tr>
<tr><td>iterator</td>
<td>Return a TextFileReader object for reading file piecemeal;<br>can also be used with the with statement.</td></tr>
<tr><td>chunksize</td>
<td>For iteration, size of file chunks.</td></tr>
<tr><td>skip_footer</td>
<td>Number of lines to ignore at end of file.</td></tr>
<tr><td>verbose</td>
<td>Print parsing info such as time spent in stages and memory usage.</td></tr>
<tr><td>encoding</td>
<td>Text encoding (e.g., "utf-8"); defaults to "utf-8" if None.</td></tr>
<tr><td>squeeze</td>
<td>If parsed data has only one column, return a Series.</td></tr>
<tr><td>thousands</td>
<td>Separator for thousands (e.g., "," or "."); default is None.</td></tr>
<tr><td>decimal</td>
<td>Decimal separator (e.g., "." or ","); default is ".".</td></tr>
<tr><td>engine</td>
<td>CSV parsing engine: "c", "python", or "pyarrow". Default is "c";<br>"pyarrow" can be faster, while "python" supports more features.</td></tr>
</table>

In [ ]:
pd.options.display.max_rows = 8 #colab overrides anything >10
pd.read_csv(github_start+"examples/ex6.csv")
#read the 1st 5 rows (not counting the header)
pd.read_csv(github_start+"examples/ex6.csv", nrows=5)

To read a file in pieces, use <i>chunksize</i> for that many rows per piece (the last piece may have fewer if it does not divide the total number of rows). chunker is an iterator where each element is a lazily loaded <i>pd.DataFrame</i> (never simultaneously stored in memory)

In [ ]:
chunker = pd.read_csv(github_start+"examples/ex6.csv", chunksize=1001)
type(chunker) #pandas.io.parsers.readers.TextFileReader
tot = pd.Series([], dtype='int64')
for piece in chunker:
    print(piece.shape)
    tot = tot.add(piece["key"].value_counts(), fill_value=0)
tot.sort_values(ascending=False)

In [ ]:
for piece in chunker: #exhausted
  print(1)

In [ ]:
chunker = pd.read_csv(github_start+"examples/ex6.csv", chunksize=1001)
print(chunker.get_chunk(6).shape) #get 1st 6 rows, overrides default chunksize
print(chunker.get_chunk().shape) #get next 1001 rows
print(next(chunker).shape) #get next 1001 rows
print(chunker.get_chunk(8638).shape) #stops at end if fewer rows left than requested

In [ ]:
data = pd.read_csv(github_start+"examples/ex5.csv")
data.to_csv("qjx.csv") #nan becomes empty string, index is 1st column with no name
!cat qjx.csv

For convenience, use <i>sys.stdout</i> to write to console instead of a file. Delimiters besides commas are allowed, use <i>sep</i> for them, <i>na_rep</i> for how to write nan, <i>columns</i> (iterator) for which columns to write and in what order (default all), <i>header</i> for whether to put column names as the top row (can also be list of strings to use), and <i>index</i> for whether to write the index as the 1st column (unnamed in top row)

In [ ]:
data.to_csv(sys.stdout, sep="|")
data.to_csv(sys.stdout, na_rep="NULL")
data.to_csv(sys.stdout, columns=["a", "b", "c"])
data.to_csv(sys.stdout, header=False, index=False)

As we see, <i>pd.read_csv</i> can load most tabular data. We outline more manual processing is needed for malformed data

In [ ]:
!curl -sL {github_start+'examples/ex7.csv'}
pd.read_csv(github_start+'examples/ex7.csv') #int64

<i>csv_reader</i> helps us iterate line by line. Each element is a list whose elements are determined based on the comma as a delimiter, with quotes treated as syntax, not data. If we did <i>for line in f</i>, then each line is a raw string (with \n)

In [ ]:
!curl -L -o ex7.csv {github_start+'examples/ex7.csv'}
f = open("ex7.csv")
reader = csv.reader(f)
for line in reader:
    print(line)
f.close()

<i>zip(*values)</i> unpacks the outer iterable, treating each inner iterable as a separate argument. This is memory intensive with big files

In [ ]:
with open("ex7.csv") as f:
    lines = list(csv.reader(f))
header, values = lines[0], lines[1:]
qjx=pd.DataFrame({h: v for h, v in zip(header, zip(*values))})
qjx.dtypes #all object

To give additional options to <i>csv.reader</i>, ie make a new format, eg with another delimiter, quoting character, or line terminator, either make a new subclass of <i>csv.Dialect</i> or pass them in directly as kwargs. In the example below, the double quote is no longer a quoting character hence appears in the data. The <i>csv</i> module cannot handle more complicated or fixed multicharacter delimiters. 1 way to do it even more manually is with <i>str.split, re.split</i>

In [ ]:
class my_dialect(csv.Dialect):
    lineterminator = "\n"
    delimiter = ";"
    quotechar = '-'
    quoting = csv.QUOTE_MINIMAL
reader = csv.reader(f, dialect=my_dialect)

In [ ]:
f = open("ex7.csv")
reader = csv.reader(f,quotechar='-')
for line in reader:
    print(line)
f.close()

more <i>csv.Dialect</i> options
<table border="1">
<tr><th>Argument</th><th>Description</th></tr>
<tr><td>delimiter</td>
<td>One-character string to separate fields; defaults to ",".</td></tr>
<tr><td>lineterminator</td>
<td>Line terminator for writing; defaults to "\r\n". Reader ignores this<br>and recognizes cross-platform line terminators.</td></tr>
<tr><td>quotechar</td>
<td>Quote character for fields with special characters (like a delimiter);<br>default is '"'.</td></tr>
<tr><td>quoting</td>
<td>Quoting convention. Options include csv.QUOTE_ALL (quote all fields),<br>csv.QUOTE_MINIMAL (only fields with special characters),<br>csv.QUOTE_NONNUMERIC, and csv.QUOTE_NONE (no quoting). Defaults to QUOTE_MINIMAL.</td></tr>
<tr><td>skipinitialspace</td>
<td>Ignore whitespace after each delimiter; default is False.</td></tr>
<tr><td>doublequote</td>
<td>How to handle quote character inside a field; if True, it is doubled<br>(see documentation for full behavior).</td></tr>
<tr><td>escapechar</td>
<td>String to escape the delimiter if quoting is set to csv.QUOTE_NONE;<br>disabled by default.</td></tr>
</table>

In [ ]:
#write csv files, accepts the same dialect and format options as csv.reader
with open("mydata.csv", "w") as f:
    writer = csv.writer(f, delimiter = " ")
    writer.writerow(("one", "two", "three"))
    writer.writerow(("1", "2", "3"))
    writer.writerow(("4", "5", "6"))
    writer.writerow(("7", "8", "9"))
!cat mydata.csv

JavaScript Object Notation (JSON), much more free form than tabular text such as CSV, is a standard format to send data by HTTP request between web browsers and other applications. It is almost valid python code, except 'null' is its null value, it forbids trailing commas, etc. Its basic types are object (dict - keys must be strings), array list, string, number, Boolean, null. Python has many libraries to read/ write JSON data, <i>json</i> is standard. <i>json.loads</i> turns a JSON string to python form, <i>json.dumps</i> does the reverse. 1 common way to make a <i>pd.DataFrame</i> is via a list of <i>dict</i> (each represents 1 row)

In [ ]:
obj = """{"name": "Wes",
 "cities_lived": ["Akron", "Nashville", "New York", "San Francisco"],
 "pet": null,
 "siblings": [{"name": "Scott", "age": 34, "hobbies": ["guitars", "soccer"]},
              {"name": "Katie", "age": 42, "hobbies": ["diving", "art"]}]}"""
result = json.loads(obj)
json.dumps(result)
pd.DataFrame(result["siblings"], columns=["name", "age"])

<i>pd.read_json</i> automatically turns JSON data to a <i>pd.DataFrame/ Series</i>. By default it assumes an array of objects, each correponding to 1 row. The <i>to_json()</i> method exports a pd object as a JSON string. By default a dict where each column name is a key, and the values are a dict with the index as keys. Use orient='records' to get a list of dicts, each corresponding to 1 row


In [ ]:
!curl -sL {github_start+'examples/example.json'}
data = pd.read_json(github_start+"examples/example.json")
data.to_json(sys.stdout)
data.to_json(sys.stdout, orient="records")

Python has many libraries to read/ write data in the ubiquitous HTML/ XML formats: <i>lxml, beautifulsoup4, html5lib</i>. The 1st is usually faster, the last 2 better handle manformed HTML/ XML. <i>pd.read_html</i> uses all of these libraries (seem to come with google colab and anaconda by default). As a demo dataset, consider bank failures from the US FDIC. <i>pd.read_html</i> by default searches for and tries to parse all tabular data in `<table>` tags, and gives a list of <i>pd.DataFrame</i>

In [ ]:
!curl -sL https://www.fdic.gov/bank/individual/failed/banklist.html | wc -l #1943 lines
!curl -sL {github_start+'examples/fdic_failed_bank_list.json'} | wc -l #1459 lines
!pip list | grep -E "lxml|beautifulsoup4|html5lib" #for conda, replace pip -> conda

In [ ]:
tables = pd.read_html("https://www.fdic.gov/bank/individual/failed/banklist.html")
len(tables) #1
failures = tables[0]
failures.head()
failures.dtypes #all object/ int64

Now that we have a <i>pd.DataFrame</i>, we can do data cleaning/ analysis which we disuss later. As an example, we find the number of bank failures per year

In [ ]:
close_timestamps = pd.to_datetime(failures["Closing Date"])
type(close_timestamps) #pd.Series
close_timestamps.dtype == "datetime64[ns]" #True dtype('<M8[ns]')
close_timestamps.dt.year.value_counts()

XML (structurally like, but more general than HTML) is also a common structured format for hierarchical, nested data and metadata.<br>
As an example we consider performance data from the New York Metropolitan Transportation Authority (MTA), found in XML files. Each train or bus service has its own file (eg Performance_MNR.xml for the Metro-North Railroad) with monthly data as a series of XML records: (INDICATOR is not a universal XML tag, it is specific to this dataset)
```
<INDICATOR>
  <INDICATOR_SEQ>373889</INDICATOR_SEQ>
  <PARENT_SEQ></PARENT_SEQ>
  <AGENCY_NAME>Metro-North Railroad</AGENCY_NAME>
  <INDICATOR_NAME>Escalator Availability</INDICATOR_NAME>
  <DESCRIPTION>Percent of the time that escalators are operational
  systemwide. The availability rate is based on physical observations performed
  the morning of regular business days only. This is a new indicator the agency
  began reporting in 2009.</DESCRIPTION>
  <PERIOD_YEAR>2011</PERIOD_YEAR>
  <PERIOD_MONTH>12</PERIOD_MONTH>
  <CATEGORY>Service Indicators</CATEGORY>
  <FREQUENCY>M</FREQUENCY>
  <DESIRED_CHANGE>U</DESIRED_CHANGE>
  <INDICATOR_UNIT>%</INDICATOR_UNIT>
  <DECIMAL_PLACES>1</DECIMAL_PLACES>
  <YTD_TARGET>97.00</YTD_TARGET>
  <YTD_ACTUAL></YTD_ACTUAL>
  <MONTHLY_TARGET>97.00</MONTHLY_TARGET>
  <MONTHLY_ACTUAL></MONTHLY_ACTUAL>
</INDICATOR>
```

In [ ]:
!curl -L -o qjx_mnr.xml {github_start+'datasets/mta_perf/Performance_MNR.xml'}
with open("qjx_mnr.xml") as f:
    parsed = objectify.parse(f) # from lxml
root = parsed.getroot() #get reference to root node

In [ ]:
data = []
skip_fields = ["PARENT_SEQ", "INDICATOR_SEQ","DESIRED_CHANGE", "DECIMAL_PLACES"]
for elt in root.INDICATOR: #generator, yields each <INDICATOR> element
    el_data = {}
    for child in elt.getchildren():
        if child.tag in skip_fields:
            continue
        el_data[child.tag] = child.pyval
    data.append(el_data)
perf = pd.DataFrame(data)
perf = pd.read_xml('qjx_mnr.xml') #same

Use the <i>pickle</i> module to serialize/ store python objects in binary format, and read them back in later. <i>pd.read_pickle()</i> and the <i>to_pickle()</i> method are really just convenience wrappers. In general, pickle files are specific to python, and meant for at most short term storage. An object pickled now might not unpickle with a later version of some library. Also, only load pickle files from trusted sources since they can execute arbitrary code

In [ ]:
frame = pd.read_csv(github_start+"examples/ex1.csv")
with open("qjx.pkl", "wb") as f:
    pickle.dump(frame, f)
#frame.to_pickle("qjx.pkl") #same
with open("qjx.pkl", "rb") as f:
    frame2 = pickle.load(f)
#frame2=pd.read_pickle("qjx.pkl") #same

In [ ]:
#example of a pickle that executes 'hidden' code
class HelloPickle:
    def __reduce__(self):
        return (print, ("Hello World",))  # runs on unpickle, cannot do print('Hello World')
with open("hello.pkl", "wb") as f:
    pickle.dump([1, None, 3, 4, HelloPickle()], f)
del HelloPickle
with open("hello.pkl", "rb") as f:
    qjx = pickle.load(f)
qjx #qjx[5] is None since print returns None

In [ ]:
#view the instructions a pickle file will follow, have AI interpret it
import pickletools
with open("hello.pkl", "rb") as f:
    data = f.read()
pickletools.dis(data)

pd allows other open source binary data formats, eg HDF5, ORC, and Apache Parquet (needs the <i>pyarrow</i> package). All are really just modern, efficient ways to store and access big data. Apache = the Apache Software Foundation (ASF), a non-profit that creates and maintains open-source software. 'Apache Parquet', 'Apache Spark' or 'Apache Kafka' refer to such projects, often used in big data / analytics pipelines

In [ ]:
!pip list | grep -E "pyarrow|openpyxl|xlrd"
#fec = pd.read_parquet('datasets/fec/fec.parquet')

pd allows reading tabular data stored in Excel (2003+) files using the <i>pd.ExcelFile</i> (faster at reading multiple sheets) class or <i>pd.read_excel()</i> function. Internally, they use the add-on packages <i>xlrd, openpyxl</i> for old-style XLS and newer XLSX files, resp. To write to excel, use the pd.ExcelWriter class for multiple sheets, or directly via the <i>to_excel()</i> method

In [ ]:
xlsx = pd.ExcelFile(github_start+"examples/ex1.xlsx")
xlsx.sheet_names #['Sheet1']
type(xlsx) #pandas.io.excel._base.ExcelFile
xlsx.parse(sheet_name="Sheet1").columns #'Unnamed: 0', 'a',etc
frame=xlsx.parse(sheet_name="Sheet1", index_col=0)
frame = pd.read_excel(github_start+"examples/ex1.xlsx", sheet_name="Sheet1") #same

In [ ]:
writer = pd.ExcelWriter("ex2.xlsx") #overwrites if exists
frame.to_excel(writer, sheet_name="Sheet1")
frame.to_excel(writer, "Sheet2")
writer.close()
pd.ExcelFile('ex2.xlsx').sheet_names #['Sheet1', 'Sheet2']
frame.to_excel("ex2.xlsx",'Sheet3') #overwrites if exists

HDF5 (hierarchical data format) is a respected file format that stores big scientific array data. Available as a C library, it has interfaces in eg Java, Julia, MATLAB, and Python. Each HDF5 file can store multiple datasets and metadata. Compared with simpler formats, HDF5 supports on-the-fly compression with many such modes, letting data with repeated patterns be stored more efficiently. HDF5 works well with datasets that do not fit in memory, efficiently reading/ writing small parts of larger arrays. pd needs the <i>tables</i> package for it. Again, many libraries can access it directly, but pd is our focus.
If processing data on remote servers (eg Amazon S3 or HDFS), a format designed for distributed storage eg Apache Parquet is better. Data analysis is often more I/O bound (reading/ writing) than CPU bound (computations), which motivates HDF5. It is still not a database (which safely allows many users to access simultaneously with concurrent reads/ writes) so its main use case is to be written once with occasional updates, but encourage frequent access/ reading. Ideal for clean, analysis ready data.<br>
The <i>pd.HDFStore</i> class works like a dict, file is made if does not already exist, else is opened for reading/ writing. The example below shows it can store many objects that need not all have the same type

In [ ]:
!pip list | grep -E "tables" #pytables in conda
store = pd.HDFStore("mydata.h5")

In [ ]:
frame = pd.DataFrame({"a": np.random.standard_normal(10)})
frame1 = pd.DataFrame({"a": np.random.standard_normal(10),
  "b": np.random.standard_normal(10)})
store["obj1"] = frame #overwrites store['obj1'] if exists
store.put("obj1", frame) #same
store["obj1_col"] = frame["a"] #pd.Series
store.close()
store

In [ ]:
store = pd.HDFStore("mydata.h5")
store["obj1"] #retrieve what was earlier stored
store.select("obj1") #same

<i>pd.HDFStore</i> has 2 storage schemas: 'fixed' (default) and 'table' (slower, but allows queries with a special syntax). The put method is an explicit version of the dict-like way to assign to `store` above

In [ ]:
store.put("obj2", frame, format="table")
#store.select("obj1", where=["index >= 6 and index <= 8"]) #TypeError
store.select("obj2", where=["index >= 6 and index <= 8"])
store.close()

The <i>to_hdf(), read_hdf()</i> methods are yet other ways to write to/ read from a .h5 file. Both need `store` to be closed. As above, the default write behavior is to append objects without overwriting the rest of the file.

In [ ]:
frame.to_hdf("mydata.h5", key="obj3", format="table")
#store.put("obj3", frame, format="table") #same
pd.read_hdf("mydata.h5", key="obj2", where=["index < 5"])
#store.select("obj2", where=["index < 5"]) #same

In [ ]:
store.close()

Many websites have public APIs providing data feeds via some format, eg JSON. Python has many ways to access these APIs, eg the <i>requests</i> package (conda install requests).
Example: find the last 30 GitHub issues for pandas via a GET HTTP request. It is good practice to call <i>raise_for_status</i> on <i>requests.get</i> output to check for HTTP errors. This output's <i>json</i> method returns a Python object with the parsed JSON data as a dict or list (depending on url)

In [ ]:
#import requests
url = "https://api.github.com/repos/pandas-dev/pandas/issues"
resp = requests.get(url)
resp.raise_for_status()
resp.json() #list of 30 dicts

In [ ]:
#extract data of interest into pd.DataFrame
resp.json()[8].keys()
pd.DataFrame(resp.json(),columns=["number", "title", "created_at","state"])

In [ ]:
headers = {"User-Agent": "python:reddit.test:v1.0 (by /u/yourusername)",
    "Accept": "application/json"}
r = requests.get('https://www.reddit.com/r/sanfrancisco/comments/1rux2g7.json', headers=headers)
#r.raise_for_status() #403 Client Error: Blocked for url
#rdtp = r.json()
!curl -o reddit_post.json https://raw.githubusercontent.com/wjasonu9/testing_git/refs/heads/master/reddit_1sz0iyx.json
with open("reddit_post.json") as qj:
    rdtp = json.load(qj) #list of 2 dicts

In [ ]:
type(rdtp[1])
rdtp[1]#.keys()

In industry, SQL-based relational DBs (eg SQL Server, PostgreSQL, MySQL) and many alternative DBs are popular. The choice of DB depends on performance, data integrity, and scalability needs.<br>
pd has functions to simplify loading a SQL query's results into a DataFrame. To illustrate, we make a SQLite3 DB using <i>sqlite3</i>

In [ ]:
#import sqlite3
query = """
CREATE TABLE test
(a VARCHAR(20), b VARCHAR(20),
 c REAL,        d INTEGER
);"""
con = sqlite3.connect("mydata.sqlite")
con.execute(query) #stage a change
con.commit() #finalize the transaction

In [ ]:
data = [("Atlanta", "Georgia", 1.25, 6),
        ("Tallahassee", "Florida", 2.6, 3),
        ("Sacramento", "California", 1.7, 5)]
con.executemany("INSERT INTO test VALUES(?, ?, ?, ?)", data)
con.commit()

In [ ]:
cursor = con.execute("SELECT * FROM test")
rows = cursor.fetchall() #standard output is list of tuples
rows==data #True

Pass this list of tuples to <i>pd.DataFrame</i>. Column names are in the cursor's description attribute. In SQLite3, this only has column names (the other fields, part of Python's DB API specification, are None. Some DB drivers give more column info)

In [ ]:
cursor.description
pd.DataFrame(rows, columns=[x[0] for x in cursor.description])
con.close() #good practice when done

The SQLAlchemy project is a Python SQL toolkit (conda install sqlalchemy) that abstracts away many common differences between SQL databases. <i>pd.read_sql</i> lets you read data from a general SQLAlchemy connection. Provided the file mydata.sqlite exists, this part is independent of the sqlite3 stuff above

In [ ]:
#import sqlalchemy as sqla
db = sqla.create_engine("sqlite:///mydata.sqlite")
pd.read_sql("SELECT * FROM test", db)

A typical workflow is to do as much preprocessing as possible in SQL, then switch to python for what SQL cannot do, eg plotting/ visuals/ ML models. Ie, make a .sql script that makes a cleaned_table in the DB, then read it into python. PostgreSQL interacts poorly with google colab, <a href=https://github.com/wjasonu9/data_jobs/blob/main/access_db.py>file</a> showing how to read from a PSQL DB

In [ ]:
!rm mydata.sqlite